# Notebook 01: Data Collection, Ingestion & Lineage Verification
## EduEquity LK: School Dropout and Education Inequality in Sri Lanka

### Research Objective
This notebook documents and executes the data acquisition workflows for:
1. **Ministry of Education (MOE) Sri Lanka:** Annual School Census data on student enrolment, dropouts, repetition, and retention rates across all 25 administrative districts (2014–2024).
2. **Department of Census and Statistics (DCS) Sri Lanka:** Child Activity Survey (CAS 2016) indicators covering child labour, schooling attendance, and primary barriers.
3. **DCS Socioeconomic Indicators:** District-level poverty headcount indices (HIES), median household income, and school infrastructure directories.
4. **Administrative District Boundaries (GeoJSON):** ADM2 district shapefiles from geoBoundaries / UN OCHA HDX for spatial choropleth mapping.


In [ ]:
import os
import sys
import json
import urllib.request
import pandas as pd
import numpy as np

# Set project root
project_root = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
sys.path.insert(0, os.path.abspath(project_root))

from src.data_pipeline import DataPipeline
from src.utils import standardize_district_name
print("Libraries loaded successfully.")


### 1. Ingestion: GeoJSON Administrative Boundaries
We fetch the standardized ADM2 administrative boundaries from the official **geoBoundaries API** for Sri Lanka to ensure boundary alignment with all 25 districts.


In [ ]:
geojson_path = os.path.join(project_root, "data/raw/sri_lanka_districts.geojson")
if os.path.exists(geojson_path):
    with open(geojson_path, "r", encoding="utf-8") as f:
        geo_data = json.load(f)
    print(f"Loaded Sri Lanka district GeoJSON successfully! Total features: {len(geo_data.get('features', []))}")
    districts = [f["properties"].get("district", f["properties"].get("shapeName")) for f in geo_data["features"]]
    print("Districts in GeoJSON:", districts[:5], "...")


### 2. Ingestion: Raw MOE & DCS Tabular Datasets
We load the raw CSV files stored in `data/raw/` and inspect their schemas and summary statistics.


In [ ]:
census_path = os.path.join(project_root, "data/raw/moe_school_census_2014_2024.csv")
socio_path = os.path.join(project_root, "data/raw/dcs_district_socioeconomic_indicators.csv")
cas_path = os.path.join(project_root, "data/raw/dcs_child_activity_survey_2016.csv")
grade_path = os.path.join(project_root, "data/raw/moe_grade_progression_dropout.csv")

df_census = pd.read_csv(census_path)
df_socio = pd.read_csv(socio_path)
df_cas = pd.read_csv(cas_path)
df_grade = pd.read_csv(grade_path)

print(f"MOE School Census records: {df_census.shape}")
print(f"DCS Socioeconomic indicators: {df_socio.shape}")
print(f"DCS Child Activity Survey records: {df_cas.shape}")
print(f"MOE Grade Progression records: {df_grade.shape}")


In [ ]:
df_census.head()
